# HumanAI Detect - Cok-Ureticili Takviye: ai_humanized_openai (Colab)

**Baglam:** Tek-uretici (Qwen2.5-7B) asiri uyumu bulundu -- canli testte Claude-yazimi
bir AI metni bile %97+ "human" cikti (bkz. proje notlari, 2026-07-19). Cozum: GPT-4o-mini
ile ai_raw sinifina 1000 yeni ornek eklendi (`data/raw/ai_raw_openai/ai_raw_openai.jsonl`,
yerel makinede API ile uretildi, kalite kontrolu yapildi -- temiz). Bu notebook, o 1000
ornegi mevcut **back-translation** humanizer'i (Helsinki-NLP OPUS-MT, TR->EN->TR) ile
`ai_humanized_openai.jsonl`'e cevirir.

**Neden Colab (bu adim gercekten GPU-bagimli, bir onceki OpenAI uretiminden FARKLI olarak):**
Yerel makinede `torch.cuda.is_available() == False` -- ceviri modeli tamamen CPU'da
calisiyor, olculen gercek hiz ~64 saniye/ornek (10 ornek = 10dk41sn). 1000 ornek icin
yerelde tahmini **~17.5 saat**. Colab'in ucretsiz GPU'su bunu muhtemelen 1-2 saate indirir.

**ONEMLI -- once kontrol et:** Bu notebook GitHub'daki kodu `git clone` ile ceker (commit
`076e184`, zaten push edildi). Eger bundan sonra yerelde YENI kod degisikligi yaparsan,
Colab'da calistirmadan once mutlaka commit+push et (bu projede daha once bu adim atlanip
bir tam GPU calistirmasi bosa gitmisti).

**Checkpoint durumu:** Yerelde zaten 15/1000 ornek uretilmis (`data/raw/ai_humanized_openai/
ai_humanized_openai.jsonl`, asagida adim 5'te yuklenecek) -- Colab bu 15'i atlayip 985
kalanla devam edecek, sifirdan baslamayacak.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')

In [ ]:
# 2. Repo klonla (guncel kod, commit 076e184+)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -3

In [ ]:
# 3. Bagimliliklari kur (backtranslate icin ozel bir sey gerekmiyor, ana [dev,colab] extra'si yeterli)
!pip install -e '.[dev,colab]' -q
!pip install sacremoses -q

In [ ]:
# 4. .env olustur (API key gerekmez, backtranslate tamamen lokal/ucretsiz bir model kullanir)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 5. Kaynak dosyalari yerel makineden yukle:
#    (a) ai_raw_openai.jsonl -- ZORUNLU, 1000 ornek (GPT-4o-mini ile uretilen kaynak)
#    (b) ai_humanized_openai.jsonl -- OPSIYONEL, 15 ornek (yerelde zaten uretilmis
#        checkpoint -- yuklenirse Colab bu 15'i ATLAYIP 985 kalanla devam eder)
from pathlib import Path
from google.colab import files

Path('data/raw/ai_raw_openai').mkdir(parents=True, exist_ok=True)
Path('data/raw/ai_humanized_openai').mkdir(parents=True, exist_ok=True)

print("ADIM 1/2: 'ai_raw_openai.jsonl' dosyasini sec (ZORUNLU)")
uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/raw/ai_raw_openai/ai_raw_openai.jsonl')

n = sum(1 for _ in open('data/raw/ai_raw_openai/ai_raw_openai.jsonl', encoding='utf-8'))
print(f'ai_raw_openai.jsonl yuklendi: {n} ornek')

In [ ]:
# 5b. (OPSIYONEL ama ONERILIR) Mevcut 15 ornekli checkpoint'i yukle -- atla, sifirdan
# baslamak istersen bu hucreyi calistirma.
from pathlib import Path
from google.colab import files

print("'ai_humanized_openai.jsonl' dosyasini sec (15 ornek, checkpoint) -- istemiyorsan bu hucreyi atla/calistirma")
uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/raw/ai_humanized_openai/ai_humanized_openai.jsonl')

p = Path('data/raw/ai_humanized_openai/ai_humanized_openai.jsonl')
if p.exists():
    n = sum(1 for _ in open(p, encoding='utf-8'))
    print(f'checkpoint yuklendi: {n} ornek, devam edilecek.')

In [ ]:
# 6. Ana isi calistir: 1000 ai_raw_openai ornegini back-translation ile humanize et.
# Checkpoint her ornekten hemen sonra yaziliyor -- oturum koparsa, ayni hucreyi tekrar
# calistirmadan once adim 8'deki (guvenlik indirme) dosyayi tekrar yukleyip devam edebilirsin.
!python scripts/humanize_ai_raw_topup.py

In [ ]:
# 7. Sonucu dogrudan Colab uzerinden yerel makineye indir
from pathlib import Path
from google.colab import files

src = Path('data/raw/ai_humanized_openai/ai_humanized_openai.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')

## Indirilen Dosyayi Yerlestirme

Onceki hucre `ai_humanized_openai.jsonl`'i tarayicinin varsayilan indirme klasorune
(`Downloads`) indirir.

Colab bittikten sonra, kendi makinende:
1. Indirilen `ai_humanized_openai.jsonl`'i `data/raw/ai_humanized_openai/ai_humanized_openai.jsonl`
   uzerine yaz (mevcut 15'lik dosyanin YERINE -- Colab zaten o 15'i icerecek sekilde devam etti).
2. `ai_raw_openai.jsonl` (1000 ornek) zaten yerel makinende hazir.
3. Bana haber ver -- kalan pipeline adimlarini (preprocess -> build_features -> ana
   veriyle birlestirip yeniden egitim ve capraz-uretici OOD tanisi) devam ettiririm.

**Oturum koparsa:** `ai_humanized_openai.jsonl`'in o ana kadarki halini indirip (adim 8'i
tekrar calistirarak, veya Colab'in son checkpoint'ini) yeni bir Colab oturumunda adim 5b
ile geri yukleyip kaldigi yerden devam edebilirsin -- sifirdan baslamana GEREK YOK.